In [1]:
// Parameters
var BATCH_MODE = "true";

The below script needs to be able to find the current output cell; this is an easy method to get it.

# Aspire AppHost : orchestrer un harness d'agent comme une ressource

Ce notebook est le **onzieme volet** de la serie Aspire (volets [01-09](README.md) et [10 — Aspire AppHost Frontend](10-Aspire-AppHost-Frontend.ipynb), livre par la PR #22370). Il ferme le residuel declare dans le [registre des axes distillables](distilled-axes-registry.md), ligne « Copilot SDK binding C# (Part 3) » : **`Aspire AppHost integration`**.

Le volet [09](09-Aspire-Harness-CopilotSdk.ipynb) a montre le harness `GitHub.Copilot.SDK` lance **a la main**, dans une cellule, en CLI. Ce volet-ci pose la question suivante :

> Que change le fait de traiter le harness comme une **ressource du graphe** plutot que comme un processus qu'on lance ? Qui decide de son port, de son repertoire de travail, de sa configuration, de son arret — le code du service, ou l'orchestrateur ?

Le lab reel vit dans [`CopilotHarness.AppHost/`](CopilotHarness.AppHost/) : un AppHost file-based (`apphost.cs` + `aspire.config.json`, meme modele que `FrontendLab/`) qui orchestre le **vrai** harness — aucun serveur de substitution, aucune reimplementation.

**Duree** : ~15-20 min. **Prerequis** : SDK .NET 10, CLI Aspire (`dotnet tool install -g aspire`), et une authentification `gh` valide (le harness herite ses credentials de la CLI GitHub ; aucune cle dans le code).

In [2]:
// Helper partage entre les cellules (.NET Interactive : un static class persiste).
// Cellule = DECLARATION SEULE : en C# top-level, aucune instruction ne peut suivre
// une declaration de type dans la meme unite de compilation.
using System.Diagnostics;
using System.IO;
using System.Linq;
using System.Net.Http;
using System.Text;
using System.Text.RegularExpressions;
using System.Threading;

public static class LabShell
{
    public static readonly string AspireCmd = Path.Combine(
        Environment.GetFolderPath(Environment.SpecialFolder.UserProfile), ".dotnet", "tools", "aspire.cmd");
    // Sur ce siege, le `dotnet` du PATH est l'hote de Program Files, qui ne voit
    // AUCUN SDK : les SDK vivent en local utilisateur. Les deux repertoires sont
    // donc resolus ici, et injectes dans chaque processus enfant (cf. section 1).
    public static readonly string DotnetRoot = Path.Combine(
        Environment.GetFolderPath(Environment.SpecialFolder.LocalApplicationData), "Microsoft", "dotnet");
    public static readonly string DotnetExe = Path.Combine(DotnetRoot, "dotnet.exe");
    public static readonly string LabDir = FindDir("CopilotHarness.AppHost");
    public static readonly string HarnessDir = FindDir("CopilotHarness.App");

    public static string FindDir(string leaf)
    {
        var dir = new DirectoryInfo(Directory.GetCurrentDirectory());
        while (dir != null && !Directory.Exists(Path.Combine(dir.FullName, "MyIA.AI.Notebooks")))
            dir = dir.Parent;
        return Path.Combine(dir!.FullName, "MyIA.AI.Notebooks", "GenAI", "Integrations-DotNet", "Aspire", leaf);
    }

    // Neutralise les chemins absolus d'une sortie d'outil avant de l'afficher
    // (meme garde que le volet 01, issue #11725) : un carnet ne publie pas
    // l'arborescence de la machine qui l'a execute.
    public static string Clean(string s) =>
        Regex.Replace(s ?? "", @"[A-Za-z]:\\[^\s""']+", "<chemin>");

    public static ProcessStartInfo Psi(string file, string[] args, string workDir, bool prefixPath = true)
    {
        var psi = new ProcessStartInfo(file)
        {
            WorkingDirectory = workDir,
            RedirectStandardOutput = true,
            RedirectStandardError = true,
            UseShellExecute = false,
            StandardOutputEncoding = Encoding.UTF8,
            StandardErrorEncoding = Encoding.UTF8,
        };
        foreach (var a in args) psi.ArgumentList.Add(a);
        psi.Environment["DOTNET_ROOT"] = DotnetRoot;
        if (prefixPath)
            psi.Environment["PATH"] = DotnetRoot + ";" + Environment.GetEnvironmentVariable("PATH");
        return psi;
    }

    public static (int code, string output) Exec(string file, string[] args, string workDir, int timeoutSec = 180)
    {
        using var p = Process.Start(Psi(file, args, workDir))!;
        var sb = new StringBuilder();
        p.OutputDataReceived += (_, e) => { if (e.Data != null) lock (sb) sb.AppendLine(e.Data); };
        p.ErrorDataReceived += (_, e) => { if (e.Data != null) lock (sb) sb.AppendLine(e.Data); };
        p.BeginOutputReadLine();
        p.BeginErrorReadLine();
        if (!p.WaitForExit(timeoutSec * 1000))
        {
            try { p.Kill(true); } catch { }
            return (-1, sb.ToString() + "\n<timeout>");
        }
        p.WaitForExit();
        return (p.ExitCode, sb.ToString());
    }

    public static Process StartBackground(string file, string[] args, string workDir)
    {
        var p = Process.Start(Psi(file, args, workDir))!;
        p.BeginOutputReadLine();
        p.BeginErrorReadLine();
        return p;
    }

    public static (int status, string body) Http(string url)
    {
        try
        {
            using var c = new HttpClient { Timeout = TimeSpan.FromSeconds(20) };
            var r = c.GetAsync(url).GetAwaiter().GetResult();
            return ((int)r.StatusCode, r.Content.ReadAsStringAsync().GetAwaiter().GetResult());
        }
        catch (Exception ex)
        {
            return (-1, ex.GetType().Name + ": " + ex.Message);
        }
    }

    public static (int status, string body) WaitHttp(string url, int tries = 40, int delayMs = 1000)
    {
        for (var i = 0; i < tries; i++)
        {
            var (st, body) = Http(url);
            if (st == 200) return (st, body);
            Thread.Sleep(delayMs);
        }
        return Http(url);
    }

    public static string UrlFromDescribe(string describe) =>
        Regex.Match(describe ?? "", @"http://localhost:\d+").Value;

    // Lecture d'un champ d'une reponse JSON minimale (evite d'exposer un chemin
    // absolu dans une sortie committee : on n'en garde que le dernier segment).
    public static string Champ(string json, string cle)
    {
        var m = Regex.Match(json ?? "", "\"" + cle + "\":\"([^\"]*)\"");
        return m.Success ? m.Groups[1].Value : "(absent)";
    }

    public static string DernierSegment(string chemin) =>
        string.IsNullOrEmpty(chemin) ? "(vide)" : chemin.Split('\\', '/').Last();
}

In [3]:
// Versions reelles des outils (garde SOTA : ce qui va s'executer est installe).
// Lecture des SDK par l'HOTE local, et par l'hote du PATH : l'ecart entre les deux
// lignes est la cause de l'echec mesure en section 1.
var vAspire = LabShell.Exec(LabShell.AspireCmd, new[] { "--version" }, LabShell.LabDir);
var sdkLocal = LabShell.Exec(LabShell.DotnetExe, new[] { "--list-sdks" }, LabShell.LabDir);
var sdkPath = LabShell.Exec("dotnet", new[] { "--list-sdks" }, LabShell.LabDir, 120);
var auth = LabShell.Exec(LabShell.DotnetExe,
    new[] { "run", "--project", LabShell.HarnessDir, "--", "auth" }, LabShell.LabDir);

string Compter(string sdkOut)
{
    var n = sdkOut.Split('\n').Count(l => l.Contains("["));
    return n == 0 ? "aucun SDK" : n + " SDK";
}

Console.WriteLine($"[aspire]            {LabShell.Clean(vAspire.output).Trim()}");
Console.WriteLine($"[SDK hote local]    {Compter(sdkLocal.output)}");
Console.WriteLine($"[SDK hote du PATH]  {Compter(sdkPath.output)}");
var lignes = auth.output.Split('\n').Where(l => l.Contains("isAuthenticated") || l.Contains("authType") || l.Contains("login"));
Console.WriteLine("[harness auth]      " + string.Join(" ", lignes.Select(l => l.Trim())));
Console.WriteLine($"[lab]               MyIA.AI.Notebooks/GenAI/Integrations-DotNet/Aspire/CopilotHarness.AppHost");

[aspire]            13.6.2+af07e33f8d1ef733645138b3f8e36067d073148d


[SDK hote local]    3 SDK


[SDK hote du PATH]  aucun SDK


[harness auth]      "isAuthenticated": true, "authType": "gh-cli", "login": "jsboige",


[lab]               MyIA.AI.Notebooks/GenAI/Integrations-DotNet/Aspire/CopilotHarness.AppHost


### Lecture de l'environnement

Deux hotes `dotnet` coexistent sur un siege ou le SDK a ete installe en portee utilisateur : celui de `Program Files` (runtimes seuls) et celui du profil (`%LOCALAPPDATA%\Microsoft\dotnet`, avec les SDK). Le PATH designe le premier, la CLI Aspire interroge le premier — et echoue avec un message qui ne dit pas ou est le SDK :

```
C# AppHost requires .NET SDK version 10.0.100 or later. Detected: (not found).
```

C'est un ecart d'environnement, pas un defaut du lab : la reparation est de **rendre l'hote local visible** (`DOTNET_ROOT` + PATH en tete pour chaque processus enfant), jamais de renoncer a l'AppHost. Le helper ci-dessus fait exactement cela, et `apphost.cs` reinjecte `DOTNET_ROOT` dans la ressource enfant — sinon `dotnet run` reussit dans un shell et echoue dans le processus orchestre.

La derniere ligne verifie l'authentification du harness avant d'aller plus loin : sans `gh` connecte, les appels modeles echoueraient plus tard, loin de leur cause.

## 1. Le lab : trois pieces, aucune reimplementation

| Fichier | Role |
|---|---|
| [`CopilotHarness.AppHost/apphost.cs`](CopilotHarness.AppHost/apphost.cs) | L'AppHost file-based : **une** ressource `copilot-harness`, ses variables d'environnement, **une** ligne d'endpoint |
| [`CopilotHarness.AppHost/aspire.config.json`](CopilotHarness.AppHost/aspire.config.json) | Pointe la CLI vers `apphost.cs` (modele SDK file-based, comme `GenAiStack.AppHost/`) |
| [`CopilotHarness.App/Program.cs`](CopilotHarness.App/Program.cs) | Le harness reel, auquel s'ajoute le mode `serve` : `/health`, `/config`, `/models`, `/ask` |

Le mode `serve` est deliberement mince : il n'implemente aucune logique d'agent. Il **expose** ce que le harness savait deja faire en CLI. C'est le point du volet — l'orchestration ne remplace pas le service, elle l'entoure.

In [4]:
Console.WriteLine("=== CopilotHarness.AppHost/apphost.cs ===");
Console.WriteLine(File.ReadAllText(Path.Combine(LabShell.LabDir, "apphost.cs")).Trim());
Console.WriteLine();
Console.WriteLine("=== fichiers du lab ===");
foreach (var f in Directory.GetFiles(LabShell.LabDir, "*", SearchOption.AllDirectories))
    Console.WriteLine("  " + Path.GetRelativePath(LabShell.LabDir, f));

=== CopilotHarness.AppHost/apphost.cs ===


#:sdk Aspire.AppHost.Sdk@13.4.6
using Aspire.Hosting;

// AppHost Aspire orchestrant le HARNESS GitHub Copilot SDK comme une ressource --
// grain #10473, residuel declare dans le registre des axes distillables
// (`Aspire/distilled-axes-registry.md`, ligne « Copilot SDK binding C# (Part 3) » :
// « Aspire AppHost integration (cf. #10857) »).
//
// Ce que ce lab demontre, et que le carnet 09 ne pouvait pas montrer : le harness
// n'est plus un processus qu'on lance a la main dans une cellule, c'est une
// RESSOURCE du graphe. L'orchestrateur decide son port, lui injecte sa
// configuration par l'environnement, suit son etat et son flux de logs -- et le
// carnet interroge la ressource a travers HTTP.
//
// Choix mesure : AddExecutable plutot que AddProject. Le harness est un projet
// .NET (CopilotHarness.App), mais un AppHost file-based (`apphost.cs`, la forme
// de cette serie) ne porte pas de ProjectReference : AddExecutable avec le vrai
// `dotnet run` lance le VRAI projet (SOTA-OK

=== fichiers du lab ===


  apphost.cs


  aspire.config.json


### Anatomie

Trois choix, chacun repayant une erreur mesuree :

| Choix | Pourquoi |
|---|---|
| `AddExecutable` plutot que `AddProject` | Un AppHost **file-based** (`apphost.cs`) ne porte pas de `ProjectReference`. `AddExecutable` lance le **vrai** `dotnet run` sur le **vrai** projet : aucune reimplementation, et le re-build reste gere par dotnet. |
| `WithHttpEndpoint(targetPort: 5099)` **sans** port hote | Une ressource executable non conteneurisee ne peut pas etre proxiee avec `Port == TargetPort` (erreur du controle DCP). Le service ecoute donc directement 5099, et l'orchestrateur expose **en plus** un proxy sur un port choisi par lui — les deux chemins sont mesures en section 3. |
| `.WithEnvironment("DOTNET_ROOT", ...)` | La ressource enfant herite de l'environnement de l'orchestrateur. Sans ce report explicite, `dotnet run` echoue dans le processus orchestre alors qu'il reussit dans le shell. |

Les variables `COPILOT_MODEL_ID` et `COPILOT_RESOURCE_NAME` ne sont lues **par aucun code du service** : elles existent pour etre restituees par `/config`. C'est ainsi que l'injection devient **observable** au lieu d'etre affirmee.

## 2. Le service existe sans l'orchestrateur

Avant de montrer l'AppHost, il faut montrer ce qu'il orchestre. Le mode `serve` est lance ici **directement**, sur un port choisi par la cellule (5100), sans Aspire.

| Route | Ce qu'elle rend |
|---|---|
| `GET /health` | `status`, `pid`, `port` |
| `GET /config` | ce que l'**environnement du processus** porte : `serviceName`, `modelId`, `resourceName`, `workingDirectory`, `port` |
| `GET /models` | le catalogue reel servi par Copilot (`ListModelsAsync`, aucun appel modele) |
| `GET /ask?prompt=...` | un tour complet, servi par l'agent |

In [5]:
// Lancement direct, sans Aspire : la cellule choisit le port ET le repertoire de travail.
// Elle le lance depuis le dossier du harness (c'est le choix de la cellule).
var directPort = "5100";
var psi = LabShell.Psi(LabShell.DotnetExe,
    new[] { "run", "--project", LabShell.HarnessDir, "--", "serve" }, LabShell.HarnessDir);
psi.Environment["PORT"] = directPort;
psi.Environment["COPILOT_MODEL_ID"] = "choisi-par-la-cellule";
psi.Environment["COPILOT_RESOURCE_NAME"] = "lancement-direct";
var svc = Process.Start(psi)!;
svc.BeginOutputReadLine();
svc.BeginErrorReadLine();

var (hStatus, hBody) = LabShell.WaitHttp($"http://127.0.0.1:{directPort}/health");
Console.WriteLine($"/health  {hStatus}  {LabShell.Clean(hBody).Trim()}");
var (cStatus, cBody) = LabShell.Http($"http://127.0.0.1:{directPort}/config");
Console.WriteLine($"/config  {cStatus}");
Console.WriteLine($"  modelId            : {LabShell.Champ(cBody, "modelId")}");
Console.WriteLine($"  resourceName       : {LabShell.Champ(cBody, "resourceName")}");
Console.WriteLine($"  repertoire courant : {LabShell.DernierSegment(LabShell.Champ(cBody, "workingDirectory"))}");
svc.Kill(true);
Console.WriteLine($"service arrete (pid {svc.Id})");

/health  200  {"status":"ok","pid":9144,"port":"5100"}


/config  200


  modelId            : choisi-par-la-cellule


  resourceName       : lancement-direct


  repertoire courant : CopilotHarness.App


service arrete (pid 17108)


### Lecture du resultat

Le service repond, et `/config` montre deja l'essentiel : `modelId` vaut exactement ce que **la cellule** a pose, et `workingDirectory` est celui du lancement. A ce stade, c'est l'appelant qui decide de tout — port, repertoire, configuration, duree de vie. Retenez ces deux valeurs : la section suivante les fait changer de main sans toucher une ligne du service.

## 3. Le meme service, orchestre

L'AppHost est lance en arriere-plan (`run --detach`), puis interroge par la CLI Aspire — `describe` pour l'etat et les URL, exactement comme dans les volets 01 et 02.

In [6]:
var launch = LabShell.Exec(LabShell.AspireCmd, new[] { "run", "--detach", "--isolated", "--non-interactive" }, LabShell.LabDir, 300);
var dash = System.Text.RegularExpressions.Regex.Match(launch.output, @"https://localhost:\d+").Value;
Console.WriteLine($"code de sortie : {launch.code}");
Console.WriteLine($"dashboard      : {dash}");
Console.WriteLine($"AppHost demarre : {launch.output.Contains("started successfully")}");

code de sortie : 0


dashboard      : https://localhost:51452


AppHost demarre : True


In [7]:
// Etat de la ressource : le harness est-il Running ET Healthy ?
var describe = LabShell.Exec(LabShell.AspireCmd, new[] { "describe" }, LabShell.LabDir, 180);
Console.WriteLine(LabShell.Clean(describe.output).Trim());
var proxyUrl = LabShell.UrlFromDescribe(describe.output);
Console.WriteLine();
Console.WriteLine($"URL de la ressource vue par l'orchestrateur : {proxyUrl}");

Scanning for running AppHosts...
┌──────────────────────────────────────────────────────────────────────────────┬────────────┬─────────┬─────────┬────────────────────────┐
│ Name                                                                         │ Type       │ State   │ Health  │ URLs                   │
├──────────────────────────────────────────────────────────────────────────────┼────────────┼─────────┼─────────┼────────────────────────┤
│ copilot-harness (https://localhost:51452/?resource=copilot-harness-gayczebn) │ Executable │ Running │ Healthy │ http://localhost:51451 │
└──────────────────────────────────────────────────────────────────────────────┴────────────┴─────────┴─────────┴────────────────────────┘


URL de la ressource vue par l'orchestrateur : http://localhost:51451


### Lecture du resultat

La ressource `copilot-harness` apparait en type **Executable**, etat **Running**, sante **Healthy**. La colonne **URLs** de ce tableau n'en liste qu'une : `http://localhost:<port>`, le **proxy** que l'orchestrateur place devant la ressource. L'ecoute directe `http://127.0.0.1:5099` ne figure pas dans ce tableau : elle est fixee par le modele de ressource (`WithHttpEndpoint`, `targetPort: 5099`). La cellule suivante interroge les deux chemins et les compare.

In [8]:
// La MEME ressource /config, posee par le proxy de l'orchestrateur puis par son
// port d'ecoute : deux chemins vers un seul processus, pas deux lancements.
var (pc, pBody) = LabShell.Http(proxyUrl + "/config");
var (dc, dBody) = LabShell.Http("http://127.0.0.1:5099/config");

Console.WriteLine($"{"champ",-20} {"via proxy " + proxyUrl,-32} {"port ressource 5099",-20}");
Console.WriteLine($"{"code HTTP",-20} {pc,-32} {dc,-20}");
foreach (var champ in new[] { "modelId", "resourceName", "port", "workingDirectory" })
{
    var gauche = champ == "workingDirectory"
        ? LabShell.DernierSegment(LabShell.Champ(pBody, champ))
        : LabShell.Champ(pBody, champ);
    var droite = champ == "workingDirectory"
        ? LabShell.DernierSegment(LabShell.Champ(dBody, champ))
        : LabShell.Champ(dBody, champ);
    Console.WriteLine($"{champ,-20} {gauche,-32} {droite,-20}");
}

champ                via proxy http://localhost:51451 port ressource 5099 


code HTTP            200                              200                 


modelId              claude-sonnet-5                  claude-sonnet-5     


resourceName         copilot-harness                  copilot-harness     


port                 5099                             5099                


workingDirectory     CopilotHarness.AppHost           CopilotHarness.AppHost


### Interpretation

Les deux colonnes du tableau precedent portent le **meme** `modelId`, `resourceName` et `port` — et c'est attendu : elles interrogent **la meme ressource orchestree**, l'une par le proxy dynamique que l'orchestrateur place devant elle, l'autre par son port d'ecoute fixe (5099). Ce n'est pas un second lancement, c'est la meme ressource vue par deux chemins. Ce tableau verifie donc que le proxy **n'altere rien** de ce que la ressource repond.

Ce qu'il ne montre pas — et c'est la comparaison qui compte — c'est l'ecart avec le **lancement direct par la cellule** (section 2), qui porte un autre `resourceName` et un autre repertoire de travail :

| | Lancement direct (section 2) | Sous AppHost (section 3) |
|---|---|---|
| qui a lance le processus | la cellule | l'orchestrateur |
| repertoire de travail | `CopilotHarness.App` | `CopilotHarness.AppHost` |

Le service n'a aucune ligne de code qui sache d'ou vient cette valeur : il lit `Environment.CurrentDirectory`. Le deplacement du repertoire est donc la **signature** de l'orchestration — **le cycle de vie, le repertoire et l'environnement appartiennent a l'orchestrateur, pas au service**.

La cellule n'affiche que le **dernier segment** du chemin : la valeur brute depend de la machine qui execute, et un carnet ne publie pas l'arborescence du poste. C'est la meme garde que `Clean`, portee au cas ou le chemin lui-meme est la mesure.

C'est la difference de nature avec un `docker-compose` : le YAML declare des conteneurs et l'orchestrateur se limite a leur cycle de vie ; ici, la ressource est un **processus arbitraire** que l'orchestrateur configure (variables injectees), adresse (proxy) et surveille (sante), sans image a construire.

In [9]:
// Catalogue reel a travers le proxy : aucun appel modele, mais une vraie session SDK.
var (mStatus, mBody) = LabShell.Http(proxyUrl + "/models");
Console.WriteLine($"/models via proxy : {mStatus}");
Console.WriteLine(LabShell.Clean(mBody).Trim());

/models via proxy : 200


{"total":13,"ids":["auto","claude-haiku-4.5","claude-haiku-5.5","claude-sonnet-5","claude-sonnet-5.5","gpt-5.3-codex","gpt-5.6-luna","gpt-5.6-terra","gpt-6-luna","grok-4.6","grok-4.7","kimi-k3","mai-code-1.1-flash"]}


### Lecture du resultat

Le catalogue traverse le proxy : la ressource orchestree n'est pas un processus factice qui repond `200` a `/health`, c'est le harness qui negocie reellement avec le runtime Copilot. La sante observee par Aspire et la capacite reelle du service sont deux choses distinctes, et ce volet les mesure separement.

## 4. Ce que l'orchestration deplace, precisement

| Question | Harness en CLI (volet 09) | Harness orchestre (ce volet) |
|---|---|---|
| Qui choisit le port ? | personne — pas de port | l'AppHost (`targetPort` + proxy) |
| Qui choisit le repertoire de travail ? | la cellule | l'AppHost |
| Qui injecte la configuration ? | l'appelant, a la main | `.WithEnvironment`, dans le modele de ressources |
| Qui sait si le service est vivant ? | personne | `describe` (Running / Healthy) |
| Qui l'arrete ? | l'appelant | `aspire stop` |
| Que faut-il ecrire dans le service ? | les modes CLI | le meme code + un mode `serve` mince |

Le gain n'est pas la ligne de code economisee : c'est que l'etat du harness devient **observable** et son cycle de vie **possedable** par autre chose que la cellule qui l'a lance.

## 5. Exercices

### Exercice 1 : une deuxieme ressource, et une dependance

Ajoute a `apphost.cs` une ressource `echo-api` (un `AddExecutable` sur un binaire de ton choix, par exemple `cmd /c "ping -t 127.0.0.1"` sous Windows) et declare que `copilot-harness` en **depend** (`.WaitFor(...)`). Verifie par `aspire describe` que l'ordre de demarrage suit la dependance.

In [10]:
// EXERCICE 1 : ajouter une ressource et une dependance dans apphost.cs, puis relancer.
// # Etape 1 : declarer var echo = builder.AddExecutable("echo-api", ...) ;
// # Etape 2 : ajouter .WaitFor(echo) sur la ressource copilot-harness ;
// # Etape 3 : relancer `aspire run --detach` et lire `aspire describe`.
// # Indice : le champ State d'une ressource en attente de dependance est visible
// #          dans describe ; c'est la preuve que l'ordre est respecte.
Console.WriteLine("Exercice a completer : modifier apphost.cs puis relancer l'AppHost.");

Exercice a completer : modifier apphost.cs puis relancer l'AppHost.


**Critere de validation** : `aspire describe` liste **deux** ressources, et la sortie de `apphost.cs` (relue par la cellule) contient bien `.WaitFor(`. Le service doit rester `Running` — une dependance mal declaree le laisse en `Waiting`.

### Exercice 2 : extraire l'URL de la ressource par son nom

`UrlFromDescribe` du helper prend la **premiere** URL de la sortie. Avec deux ressources, ce n'est plus correct : ecris `UrlPour(string describe, string nomRessource)` qui rend l'URL de la ligne portant ce nom, ou `null` si la ressource n'en expose pas encore.

In [11]:
// EXERCICE 2 : URL d'une ressource nommee, ou null si absente.
// # Etape 1 : isoler la ligne de describe dont la premiere colonne commence par <nomRessource>.
// # Etape 2 : y chercher http://localhost:\d+ ; rendre null si la ligne n'existe pas.
// # Indice : les lignes du tableau sont encadrees par le caractere de cadre utilise
// #          par `aspire describe` ; un simple Split('\n') suffit pour l'exercice.
string? UrlPour(string describe, string nomRessource)
{
    // TODO etudiant
    return null; // TODO etudiant
}
Console.WriteLine(UrlPour("| copilot-harness | Executable | Running | Healthy | http://localhost:55290 |", "copilot-harness"));


(6,7): warning CS8632: The annotation for nullable reference types should only be used in code within a '#nullable' annotations context.



**Critere de validation** : la cellule affiche l'URL pour `copilot-harness`, et `null` pour un nom inexistant. Une version qui rend toujours la premiere URL de la sortie echoue sur le second appel.

### Exercice 3 : attendre la sante sans dependre de `describe`

L'attente de la section 3 interroge `aspire describe` en boucle. Ecris `AttendreSante(string baseUrl, int maxTries)` qui sonde `/health` directement et rend le nombre d'essais necessaires, ou `-1` si le service ne repond jamais. C'est la forme robuste : elle ne depend ni du format de sortie de la CLI, ni de la disponibilite du dashboard.

In [12]:
// EXERCICE 3 : poll de /health, nombre d'essais ou -1.
// # Etape 1 : boucler maxTries fois ; a chaque tour, tester HttpClient.GetAsync(baseUrl + "/health").
// # Etape 2 : rendre l'index du premier 200 ; sortir avec -1 apres la derniere tentative.
// # Indice : Task.Delay entre deux essais, et un HttpClient unique reutilise.
async Task<int> AttendreSante(string baseUrl, int maxTries = 10)
{
    // TODO etudiant
    return -1; // TODO etudiant
}
Console.WriteLine("Exercice a completer : attendre la sante de la ressource.");

Exercice a completer : attendre la sante de la ressource.


**Critere de validation** : sur un service deja demarre, la fonction rend un nombre d'essais faible et positif ; sur un port ferme, elle rend `-1` en un temps borne. Une implementation qui rend `0` sans avoir interroge le service echoue au second cas.

In [13]:
// Nettoyage : `aspire stop` doit etre invoque DEPUIS le repertoire du lab.
// Mesure : lance depuis la racine du depot, il rend « No running AppHost found in
// the current worktree » alors que `aspire ps` liste l'AppHost — le stop est
// scope au worktree courant, pas global.
var stop = LabShell.Exec(LabShell.AspireCmd, new[] { "stop" }, LabShell.LabDir, 180);
Console.WriteLine(LabShell.Clean(stop.output).Trim());
var apres = LabShell.Exec(LabShell.AspireCmd, new[] { "ps" }, LabShell.LabDir, 120);
Console.WriteLine("--- aspire ps apres arret ---");
Console.WriteLine(LabShell.Clean(apres.output).Trim());

Scanning for running AppHosts...
Scanning for running AppHosts...
📦 Found running AppHost: apphost.cs
🛑 Sending stop signal to apphost.cs...
Stopping apphost.cs...

✅ apphost.cs stopped successfully.


--- aspire ps apres arret ---


Scanning for running AppHosts...
ℹ️ No running AppHost found. Use 'aspire run' to start one first.


### Conclusion

Le residuel de l'axe Copilot SDK est ferme : le harness est desormais **une ressource** d'un graphe Aspire, et ce volet mesure ce que cela deplace — port, repertoire, configuration, sante, arret, tous passes de l'appelant a l'orchestrateur, sans modifier la logique de l'agent.

Trois acquis transferables :

1. **Un service qui n'expose rien ne s'orchestre pas.** Le mode `serve` n'ajoute aucune intelligence ; il rend le harness adressable. C'est le prealable, et il est mince.
2. **L'injection se prouve, elle ne s'affirme pas.** Une variable que le service renvoie telle quelle transforme « l'orchestrateur configure la ressource » en fait mesure.
3. **L'ecart d'environnement se repare en tete de chaine.** Un SDK installe en portee utilisateur mais invisible du PATH fait echouer l'AppHost avec un message qui n'oriente pas ; `DOTNET_ROOT` et le PATH prefixes dans chaque processus enfant suffisent.

**Navigation** : [← 09 — le harness en CLI](09-Aspire-Harness-CopilotSdk.ipynb) · [10 — une ressource frontend →](10-Aspire-AppHost-Frontend.ipynb) · [↑ serie Aspire](README.md) · [Orleans/02 — co-host AppHost](../Orleans/02-Orleans-Aspire-CoHost.ipynb)

Grain : axe Copilot SDK (Part 3) de l'EPIC [#10473](https://github.com/jsboige/CoursIA/issues/10473) — residuel « Aspire AppHost integration » du [registre des axes](distilled-axes-registry.md), traite par l'issue [#22388](https://github.com/jsboige/CoursIA/issues/22388).